# Fluid Mechanics — Chapter 3: Visualizations

Companion notebook for Ch. 3 (*Waves*) problem-solution slides.

| Problem | Visualization | Figure |
|---------|---------------|--------|
| 3.1 | Finite-depth dispersion + particle paths | `fig_3_1_dispersion.png` |
| 3.3/3.4 | Interface modes + Rayleigh-Taylor stability | `fig_3_3_4_interface.png` |
| 3.7 | Capillary-gravity group velocity + calm region | `fig_3_7_group_vel.png` |
| 3.10 | Dispersed capillary wave packet | `fig_3_10_capillary.png` |
| 3.11 | Gaussian wave packet + spectrum | `fig_3_11_gaussian.png` |
| 3.18 | Moving-plate free surface profile | `fig_3_18_dam_break.png` |
| 3.19 | Characteristics + wave breaking | `fig_3_19_characteristics.png` |
| 3.23 | Simple wave steepening | `fig_3_23_steepening.png` |

All figures saved to `../figures/`.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from pathlib import Path

FIG_DIR = Path('../figures')
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 12,
    'axes.titlesize': 13, 'axes.labelsize': 12,
    'figure.dpi': 120,
    'axes.spines.top': False, 'axes.spines.right': False,
})
print(f'Figures → {FIG_DIR.resolve()}')


---
## Problem 3.1 — Finite-Depth Dispersion & Particle Paths

$c^2 = (g/k)\tanh(kh)$

Particle paths are ellipses with semi-axes $\propto\cosh[k(y+h)]$ (horizontal) and $\sinh[k(y+h)]$ (vertical).


In [ ]:
g = 9.81
k = np.linspace(0.01, 5, 500)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# (a) Dispersion relation c vs kh
ax = axes[0]
for kh_scale, col, lab in zip([0.5, 1, 2, 5, 20],
    ['navy','steelblue','royalblue','cornflowerblue','lightblue'],
    ['$kh=0.5$','$kh=1$','$kh=2$','$kh=5$','$kh\\to\\infty$']):
    h_val = kh_scale / k
    c2 = (g/k)*np.tanh(k*h_val)
    # Just plot at this one depth value
kh = np.linspace(0.01, 5, 400)
c_norm = np.sqrt(np.tanh(kh)/kh)  # c/sqrt(gh)
c_deep = 1/np.sqrt(kh)            # deep water limit
c_shallow = np.ones_like(kh)      # shallow water limit
ax.plot(kh, c_norm, 'b-', lw=2.5, label='Exact: $c/\\sqrt{gh}$')
ax.plot(kh, c_deep, 'r--', lw=1.5, alpha=0.7, label='Deep: $c=\\sqrt{g/k}$')
ax.axhline(1, color='green', lw=1.5, ls=':', alpha=0.7, label='Shallow: $c=\\sqrt{gh}$')
ax.set_xlabel('$kh$'); ax.set_ylabel('$c/\\sqrt{gh}$')
ax.set_title('(a) Dispersion relation'); ax.legend(fontsize=9)
ax.set_xlim(0, 5); ax.set_ylim(0, 1.5); ax.grid(True, alpha=0.2)

# (b) Phase and group velocity vs kh
ax = axes[1]
kh = np.linspace(0.01, 5, 400)
c2 = np.tanh(kh)/kh  # normalised c^2
c_ph = np.sqrt(c2)  # c/sqrt(gh)
# cg = d(omega)/dk = c/2 * (1 + 2kh/sinh(2kh))
cg_norm = 0.5*c_ph * (1 + 2*kh/np.sinh(2*kh))
ax.plot(kh, c_ph, 'b-', lw=2.5, label='Phase $c$')
ax.plot(kh, cg_norm, 'r-', lw=2.5, label='Group $c_g$')
ax.axhline(0.5, color='gray', lw=1, ls='--', alpha=0.6, label='Deep limit $c_g=c/2$')
ax.set_xlabel('$kh$'); ax.set_ylabel('Speed / $\\sqrt{gh}$')
ax.set_title('(b) Phase vs group velocity'); ax.legend(fontsize=9)
ax.set_xlim(0, 5); ax.grid(True, alpha=0.2)

# (c) Particle paths at different depths
ax = axes[2]
k0 = 1.0; h0 = 2.0; A = 0.3
depths = np.array([0, -0.5, -1.0, -1.5, -h0+0.01])
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(depths)))
for y0, col in zip(depths, colors):
    a_horiz = A * np.cosh(k0*(y0+h0)) / np.sinh(k0*h0)
    a_vert  = A * np.sinh(k0*(y0+h0)) / np.sinh(k0*h0)
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(a_horiz*np.cos(theta), y0+a_vert*np.sin(theta),
            color=col, lw=1.8)
    ax.plot(0, y0, 'o', color=col, ms=4)
ax.axhline(-h0, color='brown', lw=2.5)
ax.axhline(0,   color='b',     lw=1.5, ls='--', alpha=0.5)
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title('(c) Particle paths (ellipses)\n deeper = flatter')
ax.set_aspect('equal'); ax.set_xlim(-0.6, 0.6)
ax.text(-0.55, -h0+0.05, 'Seabed', fontsize=9, color='brown')

plt.suptitle('Problem 3.1 — Finite-Depth Surface Waves', fontsize=13, y=1.01)
plt.tight_layout(pad=1.2)
plt.savefig(FIG_DIR/'fig_3_1_dispersion.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: fig_3_1_dispersion.png')


---
## Problems 3.3 & 3.4 — Interface Modes & Rayleigh–Taylor Stability

$\omega_N^2 = \frac{N\pi}{a(\rho_1+\rho_2)}\left[(\rho_1-\rho_2)g + T\frac{N^2\pi^2}{a^2}\right]$

Unstable ($\omega_N^2<0$) when $(\rho_2-\rho_1)g > \pi^2 T/a^2$.


In [ ]:
rho1 = 1000; rho2 = 900; g = 9.81; T = 0.07; a = 0.1
N_modes = np.arange(1, 6)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# (a) Interface mode shapes
ax = axes[0]
x = np.linspace(0, a, 300)
colors = plt.cm.tab10(np.linspace(0, 0.5, len(N_modes)))
for N, col in zip(N_modes, colors):
    eta = np.cos(N*np.pi*x/a)
    ax.plot(x*100, 0.1*eta + N*0.3, color=col, lw=2, label=f'$N={N}$')
    ax.axhline(N*0.3, color='gray', lw=0.5, ls=':', alpha=0.5)
ax.axvline(0,    color='k', lw=2)
ax.axvline(a*100, color='k', lw=2)
ax.set_xlabel('$x$ (cm)'); ax.set_ylabel('Mode (offset)')
ax.set_title('(a) Normal mode shapes $\\cos(N\\pi x/a)$')
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(-1, a*100+1); ax.grid(True, alpha=0.2)

# (b) Stability diagram: omega^2 vs N for RT instability
ax = axes[1]
a_vals = np.linspace(0.005, 0.15, 300)  # container width in m
rho1_rt = 900; rho2_rt = 1000           # RT: heavier on top
for N, col in zip([1, 2, 3], ['tomato','orange','gold']):
    k_N = N*np.pi/a_vals
    omega2 = k_N/((rho1_rt+rho2_rt)) * (-(rho2_rt-rho1_rt)*g + T*k_N**2)
    ax.plot(a_vals*100, omega2, color=col, lw=2.2, label=f'$N={N}$')
ax.axhline(0, color='k', lw=1.5)
ax.fill_between(a_vals*100, 0,
    np.minimum(0, np.array([min(k/((rho1_rt+rho2_rt))*(-(rho2_rt-rho1_rt)*g+T*(k)**2)
    for k in [np.pi/a] ) for a in a_vals])),
    alpha=0.08, color='red')
# Critical width
a_crit = np.pi * np.sqrt(T/((rho2_rt-rho1_rt)*g))
ax.axvline(a_crit*100, color='red', lw=1.5, ls='--',
           label=f'$a_c={a_crit*100:.1f}$ cm (N=1 stable)')
ax.set_xlabel('Container width $a$ (cm)')
ax.set_ylabel('$\\omega_N^2$ (rad²/s²)')
ax.set_title('(b) Rayleigh–Taylor: $\\omega_N^2$ vs $a$\n'
             'Negative = unstable ($\\rho_2>\\rho_1$)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

plt.suptitle('Problems 3.3 & 3.4 — Interface Modes & RT Instability', fontsize=13, y=1.01)
plt.tight_layout(pad=1.2)
plt.savefig(FIG_DIR/'fig_3_3_4_interface.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: fig_3_3_4_interface.png')
print(f'Critical width for RT: a_c = {a_crit*100:.2f} cm')


---
## Problem 3.7 — Group Velocity of Capillary-Gravity Waves

$c_g^{\min} = (4gT/\rho)^{1/4}$ — speed of the expanding calm region.


In [ ]:
g = 9.81; rho = 1e3; T_st = 0.074
k = np.linspace(1, 2000, 2000)

omega = np.sqrt(g*k + T_st/rho * k**3)
c_ph  = omega/k
cg    = (g + 3*T_st/rho*k**2) / (2*omega)

k_min = (rho*g/T_st)**0.5
cg_min = (4*g*T_st/rho)**0.25
lam_min = 2*np.pi/k_min

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

ax = axes[0]
ax.semilogx(2*np.pi/k*100, c_ph*100, 'b-', lw=2.2, label='Phase speed $c$')
ax.semilogx(2*np.pi/k*100, cg*100, 'r-', lw=2.2, label='Group speed $c_g$')
ax.axhline(cg_min*100, color='green', lw=1.5, ls='--',
           label=f'$c_g^{{\\min}}={cg_min*100:.1f}$ cm/s')
ax.axvline(lam_min*100, color='purple', lw=1.5, ls=':',
           label=f'$\\lambda^*={lam_min*100:.2f}$ cm')
ax.axvline(4.5, color='orange', lw=1.5, ls='-.',
           label='Observed $\\lambda\\approx4.5$ cm')
ax.set_xlabel('Wavelength $\\lambda$ (cm, log scale)')
ax.set_ylabel('Speed (cm/s)')
ax.set_title('(a) Phase & group velocity\nCapillary-gravity waves')
ax.legend(fontsize=8); ax.grid(True, alpha=0.2)
ax.set_xlim(0.1, 100); ax.set_ylim(0, 100)

ax = axes[1]
t_vals = [0.5, 1.0, 2.0, 3.0]
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(t_vals)))
for t, col in zip(t_vals, colors):
    r_calm = cg_min * t
    theta = np.linspace(0, 2*np.pi, 300)
    ax.plot(r_calm*np.cos(theta), r_calm*np.sin(theta),
            color=col, lw=2, label=f'$t={t}$ s')
    ax.fill(r_calm*np.cos(theta), r_calm*np.sin(theta),
            alpha=0.06, color=col)
ax.plot(0, 0, 'ko', ms=8, zorder=5)
ax.text(0.05, 0.05, 'Stone\nimpact', fontsize=9, ha='left')
ax.set_aspect('equal')
ax.set_xlabel('$x$ (m)'); ax.set_ylabel('$y$ (m)')
ax.set_title(f'(b) Calm region radius $= c_g^{{\\min}}\\cdot t$\n'
             f'$c_g^{{\\min}}={cg_min*100:.1f}$ cm/s')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

plt.suptitle('Problem 3.7 — Stone in a Pond: Group Velocity', fontsize=13, y=1.01)
plt.tight_layout(pad=1.2)
plt.savefig(FIG_DIR/'fig_3_7_group_vel.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'cg_min = {cg_min*100:.2f} cm/s,  lambda* = {lam_min*100:.3f} cm')


---
## Problem 3.10 — Dispersed Capillary Wave Packet

$\eta(x,t)\approx A(x,t)\cos\!\left(\frac{4\rho x^3}{27Tt^2}+\varepsilon\right)$


In [ ]:
rho = 1e3; T_st = 0.074
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

x = np.linspace(0.01, 1.5, 1000)
t_vals = [1.0, 2.0, 4.0]
colors = plt.cm.plasma(np.linspace(0.2, 0.85, len(t_vals)))

# (a) Wave pattern at different times
ax = axes[0]
for t, col in zip(t_vals, colors):
    phase = 4*rho*x**3 / (27*T_st*t**2)
    # Slowly varying amplitude (Airy-function type envelope)
    k_local = 2*rho*x**2 / (9*T_st*t**2)  # local wavenumber
    A = (k_local + 0.1)**(-0.25)           # amplitude ~ k^{-1/4}
    A /= A.max()
    eta = A * np.cos(phase)
    ax.plot(x*100, eta + t*1.5, color=col, lw=1.5, label=f'$t={t}$ s')
ax.set_xlabel('$x$ (cm)'); ax.set_ylabel('$\\eta$ (offset by $t$)')
ax.set_title('(a) Dispersed capillary wave\n'
             r'$\eta\propto\cos(4\rho x^3/27Tt^2)$')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

# (b) Local wavelength vs x at fixed t
ax = axes[1]
t0 = 2.0
k_local = 2*rho*x**2 / (9*T_st*t0**2)
lam_local = 2*np.pi/k_local * 100  # cm
ax.plot(x*100, lam_local, 'b-', lw=2.5)
ax.set_xlabel('$x$ (cm)'); ax.set_ylabel('Local $\\lambda$ (cm)')
ax.set_title(f'(b) Local wavelength at $t={t0}$ s\n'
             r'$\lambda = 2\pi/k \propto x^{-2}$')
ax.set_ylim(0, 30); ax.grid(True, alpha=0.2)

plt.suptitle('Problem 3.10 — Dispersed Capillary Wave Packet', fontsize=13, y=1.01)
plt.tight_layout(pad=1.2)
plt.savefig(FIG_DIR/'fig_3_10_capillary.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: fig_3_10_capillary.png')


---
## Problem 3.11 — Gaussian Wave Packet

$\eta(x,0) = a_0(\pi/\sigma)^{1/2}e^{-x^2/4\sigma}e^{ik_0 x}$


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
k0 = 10.0

for ax, sigma, label in zip(axes, [0.5, 2.0, 8.0],
                             ['Few crests\n$k_0\\sqrt{\\sigma}\\approx7$',
                              'Moderate\n$k_0\\sqrt{\\sigma}\\approx14$',
                              'Many crests\n$k_0\\sqrt{\\sigma}\\approx28$']):
    x = np.linspace(-4*np.sqrt(sigma), 4*np.sqrt(sigma), 1000)
    k = np.linspace(k0-5/np.sqrt(sigma), k0+5/np.sqrt(sigma), 500)

    # Real part of wave packet
    eta = np.exp(-x**2/(4*sigma)) * np.cos(k0*x)
    envelope = np.exp(-x**2/(4*sigma))

    ax.plot(x, eta, 'b-', lw=1.5, alpha=0.85, label=r'$\eta(x,0)$')
    ax.plot(x,  envelope, 'r--', lw=1.8, alpha=0.7, label='Envelope')
    ax.plot(x, -envelope, 'r--', lw=1.8, alpha=0.7)

    # Spectrum inset
    a_k = np.exp(-sigma*(k-k0)**2)
    ax2 = ax.inset_axes([0.62, 0.55, 0.36, 0.38])
    ax2.plot(k, a_k, 'g-', lw=1.5)
    ax2.set_title('$a(k)$', fontsize=8)
    ax2.set_xlabel('$k$', fontsize=7)
    ax2.tick_params(labelsize=7)
    ax2.axvline(k0, color='k', lw=0.8, ls='--')

    ax.set_xlabel('$x$'); ax.set_ylabel(r'$\eta$')
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.2)

plt.suptitle('Problem 3.11 — Gaussian Wave Packet ($k_0=10$, varying $\\sigma$)',
             fontsize=13, y=1.01)
plt.tight_layout(pad=1.2)
plt.savefig(FIG_DIR/'fig_3_11_gaussian.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: fig_3_11_gaussian.png')


---
## Problem 3.18 — Moving Plate: Free Surface Profile (Fig. 3.26)

Three regions: undisturbed, rarefaction fan, uniform behind plate.


In [ ]:
g = 9.81; h0 = 1.0; c0 = np.sqrt(g*h0); V = 0.8*c0
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, t in zip(axes, [1.0, 2.5]):
    x_left  = -c0*t
    x_right = V*t
    x_mid_r = (1.5*V - c0)*t

    x = np.linspace(-1.5*c0*t, V*t*1.1, 800)
    h = np.zeros_like(x)
    u = np.zeros_like(x)

    for i, xi in enumerate(x):
        if xi <= x_left:
            h[i] = h0; u[i] = 0
        elif xi <= x_mid_r:
            c_val = (2*c0 - xi/t)/3
            h[i] = c_val**2/g
            u[i] = 2*(c_val - c0)/(-1) + 0  # = 2/3*(c0 + xi/t)
        elif xi <= x_right:
            c_val = c0 - V/2
            h[i] = c_val**2/g
            u[i] = V
        else:
            h[i] = 0

    # Fill under free surface
    ax.fill_between(x, 0, h, alpha=0.25, color='steelblue')
    ax.plot(x, h, 'b-', lw=2.5, label='Free surface $h(x,t)$')
    ax.axhline(h0, color='gray', lw=1, ls='--', alpha=0.5, label=f'$h_0={h0}$ m')
    # Boundary markers
    ax.axvline(x_left,  color='r',   lw=1.5, ls=':', alpha=0.7, label=f'$x=-c_0t$')
    ax.axvline(x_mid_r, color='orange', lw=1.5, ls=':', alpha=0.7, label=f'$x=(\\frac{{3}}{{2}}V-c_0)t$')
    ax.axvline(x_right, color='green', lw=2.5, label=f'Plate $x=Vt$')
    ax.axhline(0, color='k', lw=2)
    # Region labels
    ax.text((x_left-1.5*c0*t)/2, h0*0.6, '(i)\nUndisturbed', ha='center', fontsize=8)
    ax.text((x_left+x_mid_r)/2, h0*0.3, '(ii)\nRarefaction', ha='center', fontsize=8)
    ax.text((x_mid_r+x_right)/2, h0*0.3, '(iii)\nUniform', ha='center', fontsize=8)
    ax.set_xlabel('$x$ (m)'); ax.set_ylabel('$h$ (m)')
    ax.set_title(f'Free surface at $t={t}$ s  ($V={V:.2f}$ m/s, $c_0={c0:.2f}$ m/s)')
    ax.legend(fontsize=8, loc='upper right'); ax.grid(True, alpha=0.2)

plt.suptitle('Problem 3.18 — Moving Plate: Dam-Break Variant', fontsize=13, y=1.01)
plt.tight_layout(pad=1.2)
plt.savefig(FIG_DIR/'fig_3_18_dam_break.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: fig_3_18_dam_break.png')


---
## Problem 3.19 — Characteristics and Wave Breaking

Characteristics $x = x_0 + g(x_0)t$; breaking at $t_c = -1/g'(x_0)$.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

x0_vals = np.linspace(-3, 3, 20)

# (a) g'(x) > 0: diverging characteristics
ax = axes[0]
def g_pos(x): return np.tanh(x) + 1.5   # g' > 0
t = np.linspace(0, 1.5, 100)
colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(x0_vals)))
for x0, col in zip(x0_vals, colors):
    x_char = x0 + g_pos(x0)*t
    ax.plot(x_char, t, color=col, lw=1.2, alpha=0.8)
ax.set_xlabel('$x$'); ax.set_ylabel('$t$')
ax.set_title('(a) $g\'(x)>0$: characteristics diverge\nSmooth solution persists')
ax.set_xlim(-5, 8); ax.set_ylim(0, 1.5)
ax.invert_yaxis()
ax.grid(True, alpha=0.2)

# (b) g'(x) < 0: converging characteristics → wave breaking
ax = axes[1]
def g_neg(x): return -np.tanh(x) + 0.5   # g' < 0 near x=0
# Breaking time at x0=0: t_c = -1/g'(0)
from scipy.misc import derivative
gp_0 = derivative(g_neg, 0.0, dx=1e-5)
t_break = -1/gp_0
t = np.linspace(0, t_break*1.1, 100)
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(x0_vals)))
for x0, col in zip(x0_vals, colors):
    x_char = x0 + g_neg(x0)*t
    ax.plot(x_char, t, color=col, lw=1.2, alpha=0.8)
ax.axhline(t_break, color='k', lw=2, ls='--',
           label=f'Breaking $t_c={t_break:.2f}$')
ax.set_xlabel('$x$'); ax.set_ylabel('$t$')
ax.set_title(f'(b) $g\'(x)<0$: characteristics converge\nWave breaks at $t_c={t_break:.2f}$')
ax.set_xlim(-5, 5); ax.set_ylim(0, t_break*1.1)
ax.invert_yaxis()
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

plt.suptitle('Problem 3.19 — Characteristics and Wave Breaking', fontsize=13, y=1.01)
plt.tight_layout(pad=1.2)
plt.savefig(FIG_DIR/'fig_3_19_characteristics.png', dpi=150, bbox_inches='tight')
plt.show(); print(f'Saved: fig_3_19_characteristics.png  t_break={t_break:.3f}')


---
## Problem 3.23 — Simple Wave Steepening

$u(x,0)=\tfrac{1}{2}U[1-\tanh(x/L)]$; solution breaks at $t^*=4L/(\gamma+1)U$.


In [ ]:
gamma = 1.4; U = 1.0; L = 1.0; a0 = 3.0
t_break = 4*L / ((gamma+1)*U)
print(f't* = {t_break:.3f}')

x = np.linspace(-6, 6, 800)

def u0(x): return 0.5*U*(1 - np.tanh(x/L))

def u_at_t(t, x_grid):
    """Solve x = x0 + c(u0(x0))*t implicitly for each x."""
    u_out = np.zeros_like(x_grid)
    x0_arr = np.linspace(-10, 10, 4000)
    c_x0 = 0.5*(gamma+1)*u0(x0_arr) + a0
    x_arr = x0_arr + c_x0*t
    for i, xi in enumerate(x_grid):
        idx = np.argmin(np.abs(x_arr - xi))
        u_out[i] = u0(x0_arr[idx])
    return u_out

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# (a) Evolving u profiles
ax = axes[0]
t_plot = [0, 0.3*t_break, 0.7*t_break, 0.95*t_break]
colors = plt.cm.YlOrRd(np.linspace(0.2, 0.9, len(t_plot)))
ax.plot(x, u0(x), color=colors[0], lw=2.5, label='$t=0$')
for t_val, col in zip(t_plot[1:], colors[1:]):
    u_t = u_at_t(t_val, x)
    ax.plot(x, u_t, color=col, lw=2,
            label=f'$t={t_val/t_break:.2f}t^*$')
ax.axvline(0, color='gray', lw=0.8, ls=':', alpha=0.5)
ax.set_xlabel('$x$'); ax.set_ylabel('$u$')
ax.set_title(f'(a) Steepening profile\nBreaks at $t^*={t_break:.2f}$')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

# (b) Characteristics in x-t plane
ax = axes[1]
t_arr = np.linspace(0, t_break*1.05, 200)
x0_ch = np.linspace(-5, 5, 30)
colors_ch = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(x0_ch)))
for x0, col in zip(x0_ch, colors_ch):
    c_val = 0.5*(gamma+1)*u0(x0) + a0
    ax.plot(x0 + c_val*t_arr, t_arr, color=col, lw=1.0, alpha=0.75)
ax.axhline(t_break, color='k', lw=2, ls='--',
           label=f'$t^*={t_break:.2f}$')
ax.set_xlabel('$x$'); ax.set_ylabel('$t$')
ax.set_title('(b) Characteristic lines\nConverge at $t^*$')
ax.set_xlim(-6, 8); ax.set_ylim(0, t_break*1.1)
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

plt.suptitle('Problem 3.23 — Simple Wave Steepening and Shock Formation',
             fontsize=13, y=1.01)
plt.tight_layout(pad=1.2)
plt.savefig(FIG_DIR/'fig_3_23_steepening.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: fig_3_23_steepening.png')


---
## Download all figures

In [ ]:
import zipfile
fig_files = [
    'fig_3_1_dispersion.png', 'fig_3_3_4_interface.png',
    'fig_3_7_group_vel.png',  'fig_3_10_capillary.png',
    'fig_3_11_gaussian.png',  'fig_3_18_dam_break.png',
    'fig_3_19_characteristics.png', 'fig_3_23_steepening.png',
]
zip_path = Path('ch03_figures.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in fig_files:
        fpath = FIG_DIR / fname
        if fpath.exists():
            zf.write(fpath, fname)
            print(f'  added: {fname}  ({fpath.stat().st_size/1024:.0f} KB)')
        else:
            print(f'  MISSING: {fname}')
print(f'\nZip: {zip_path}  ({zip_path.stat().st_size/1024:.0f} KB)')
try:
    from google.colab import files
    files.download(str(zip_path))
    print('Download started.')
except ImportError:
    print(f'Not in Colab — saved locally: {zip_path.resolve()}')
